# 사용자실험: 기술통계

사용자실험에서 수집한 7개 설문 종속변수의 평균과 표준편차를 전체, UI 유형, 숙련도 및 UI×숙련도 집단별로 계산합니다. 
마지막에는 논문에 사용할 수 있는 `M (SD)` 형식의 표를 출력합니다.

> 인지부하는 점수가 높을수록 부하가 크므로 다른 종속변수와 해석 방향이 반대입니다.


## 분석 순서

1. 원자료와 분석 문항을 준비합니다.
2. 참가자별 종속변수 점수를 계산합니다.
3. 사전 AI 리터러시로 숙련도를 분류합니다.
4. 집단별 인원과 결측값을 검증합니다.
5. 평균과 표준편차를 계산합니다.
6. 논문용 기술통계표를 만듭니다.


## 1. 데이터 준비

### 1-1. 라이브러리와 원자료 불러오기

엑셀의 `병합 데이터` 시트를 읽고 참가자 수와 열 수만 확인합니다.


In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

import numpy as np
import pandas as pd
from IPython.display import display

survey = common.load_survey()

print(f"자료 크기: {survey.shape[0]}명 × {survey.shape[1]}개 열")


### 1-2. 종속변수와 숙련도 문항 정의

7개 종속변수의 문항 번호와 숙련도 분류에 사용할 사전 AI 리터러시 6문항을 정의합니다.


In [ ]:
DV_ITEMS = common.DV_ITEMS

LITERACY_ITEMS = common.LITERACY_ITEMS

item_map = pd.DataFrame({
    "종속변수": list(DV_ITEMS),
    "사용 문항": [", ".join(items) for items in DV_ITEMS.values()],
})
display(item_map)


### 1-3. 문항 열 찾기와 평균 계산 함수

`find_columns()`는 문항 번호에 해당하는 실제 엑셀 열을 찾고, `average_score()`는 참가자별 문항 평균을 계산합니다. 문항 연결이 잘못되거나 결측값이 있으면 분석을 중단합니다.


In [ ]:
find_columns = partial(common.find_columns, survey)
average_score = partial(common.average_score, survey)


---

## 2. 참가자별 분석 점수 만들기

### 2-1. 7개 종속변수 점수 계산

여러 문항으로 구성된 변수는 참가자별 문항 평균을 사용합니다.

$$Y_{ic}=\frac{1}{k_c}\sum_{j=1}^{k_c}Q_{icj}$$

아래 미리보기의 한 행이 참가자 한 명의 7개 분석 점수입니다.


In [ ]:
scores = common.make_scores(survey)

print(f"점수표 크기: {scores.shape[0]}명 × {scores.shape[1]}개 종속변수")
display(scores.head().round(2))


### 2-2. UI 유형과 숙련도 변수 만들기

사전 AI 리터러시는 6-1~6-6의 평균입니다. 평균이 6.0 이상이면 숙련자, 미만이면 초보자로 분류합니다. 원자료의 `LLM 경험` 열은 이 분류에 사용하지 않습니다.


In [ ]:
groups = common.make_design(survey)
literacy_pre = groups["literacy_pre"]
display(groups.head())
print(f"literacy_pre 중앙값: {groups['literacy_pre'].median():.4f}")


### 2-3. 분석 전 검증과 집단 인원 확인

참가자 수, 종속변수 수, 결측값을 확인한 다음 UI×숙련도 교차표를 출력합니다. 최신 자료의 네 셀은 각각 10명입니다.


In [ ]:
assert len(survey) == 40, "참가자 수가 40명이 아닙니다."
assert scores.shape == (40, 7), "종속변수 점수표의 크기가 예상과 다릅니다."
assert scores.notna().all().all(), "종속변수 점수에 결측값이 있습니다."
assert groups["literacy_pre"].notna().all(), "리터러시 점수에 결측값이 있습니다."

classification = pd.crosstab(groups["UI"], groups["숙련도"]).reindex(
    index=["A2UI", "Baseline"],
    columns=["초보자", "숙련자"],
)
assert classification.shape == (2, 2)
assert classification.notna().all().all() and classification.gt(1).all().all()
assert classification.to_numpy().sum() == len(survey)

classification_with_total = classification.copy()
classification_with_total["합계"] = classification_with_total.sum(axis=1)
classification_with_total.loc["합계"] = classification_with_total.sum(axis=0)
display(classification_with_total.rename_axis("UI 유형"))


## 3. 평균과 표준편차 계산

### 3-1. 계산식과 표시 함수

$$M=\frac{1}{n}\sum_{i=1}^{n}x_i,\qquad SD=\sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-M)^2}$$

논문 표에서 자주 사용하는 `M (SD)` 형식으로 값을 표시합니다. 표준편차는 `ddof=1`인 표본표준편차입니다.


In [ ]:
def mean_sd(values):
    mean = values.mean()
    standard_deviation = values.std(ddof=1)
    return f"{mean:.2f} ({standard_deviation:.2f})"



### 3-2. 집단별 요약 함수

`summarize()`는 집단 선택 조건을 받아 모든 종속변수의 `M (SD)`를 한 표로 만듭니다. 먼저 전체, UI 유형, 숙련도 집단을 정의합니다.


In [ ]:
def summarize(group_masks):
    return pd.DataFrame({
        group_name: {
            variable: mean_sd(scores.loc[mask, variable])
            for variable in DV_ITEMS
        }
        for group_name, mask in group_masks.items()
    })


main_group_masks = {
    "전체 (N=40)": pd.Series(True, index=scores.index),
    "Baseline (n=20)": groups["UI"].eq("Baseline"),
    "A2UI (n=20)": groups["UI"].eq("A2UI"),
    "초보자 (n=20)": groups["숙련도"].eq("초보자"),
    "숙련자 (n=20)": groups["숙련도"].eq("숙련자"),
}

display(pd.Series({name: int(mask.sum()) for name, mask in main_group_masks.items()}, name="n").to_frame())


## 4. 논문용 기술통계표

### 4-1. 전체·UI 유형·숙련도별 기술통계

논문의 기본 기술통계표로 사용할 수 있도록 종속변수를 행에, 전체와 주요 집단을 열에 배치합니다. 통계적 유의성은 이원분산분석 노트북의 F값과 p값으로 판단합니다.


In [ ]:
publication_descriptive_table = summarize(main_group_masks)
publication_descriptive_table.index.name = "종속변수"

display(publication_descriptive_table)
print("주. 값은 M (SD)이다. 인지부하는 점수가 높을수록 부하가 큰 것을 의미한다.")


### 4-2. UI 유형 × 숙련도 셀별 기술통계

상호작용을 해석하거나 부록에 제시할 수 있도록 2×2 설계의 네 셀을 별도 표로 출력합니다. 표 제목의 `n`은 선택된 실제 참가자 수에서 자동 계산합니다.


In [ ]:
cell_group_masks = {}

for ui in ["A2UI", "Baseline"]:
    for proficiency in ["초보자", "숙련자"]:
        mask = groups["UI"].eq(ui) & groups["숙련도"].eq(proficiency)
        label = f"{ui}·{proficiency} (n={int(mask.sum())})"
        cell_group_masks[label] = mask

publication_cell_table = summarize(cell_group_masks)
publication_cell_table.index.name = "종속변수"

display(publication_cell_table)
print("주. 값은 M (SD)이다.")


### 기술통계 표 재사용

평균과 표본표준편차는 앞에서 계산한 결과를 재사용합니다. 표와 그림은 표시 형식만 바꿉니다.

In [ ]:
# Reuse the computed descriptive table; only the column layout changes.
table_4_3 = publication_descriptive_table.copy()
table_4_3.columns = pd.MultiIndex.from_tuples([
    ("전체", "Overall (N=40)"), ("UI type", "Baseline (n=20)"),
    ("UI type", "A2UI (n=20)"), ("숙련도", "초보자 (n=20)"), ("숙련도", "숙련자 (n=20)")])
table_4_3.index.name = "사후 설문"


---

## 5. 논문용 시각화: 전체 평균과 95% 신뢰구간

7점 척도에서 각 종속변수의 전체 평균과 95% 신뢰구간을 함께 표시합니다. 변수들의 전반적인 수준과 추정의 불확실성을 한눈에 비교할 수 있습니다. 점선은 척도의 중간값인 4점입니다.


In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

KOREAN_FONT = common.configure_plots()
FIGURE_DIR = common.FIGURE_DIR


**그림 2. 사용자실험 설문 종속변수의 전체 평균.** 점은 평균, 오차막대는 95% 신뢰구간이며 점선은 7점 척도의 중간값(4점)이다. 인지부하는 높은 점수가 더 큰 부하를 의미한다.


In [ ]:
mean_values = scores.mean()
standard_errors = scores.sem(ddof=1)
critical_t = stats.t.ppf(.975, df=len(scores) - 1)
ci_margins = critical_t * standard_errors

plot_order = list(reversed(list(DV_ITEMS)))
y_positions = np.arange(len(plot_order))

fig, ax = plt.subplots(figsize=(7.2, 4.7))
ax.errorbar(
    mean_values[plot_order],
    y_positions,
    xerr=ci_margins[plot_order],
    fmt="o",
    color="black",
    ecolor="0.40",
    elinewidth=1.4,
    capsize=3,
)

for y_position, variable in zip(y_positions, plot_order):
    ax.text(mean_values[variable] + ci_margins[variable] + .08, y_position, f"{mean_values[variable]:.2f}", va="center", fontsize=9)

ax.axvline(4, color="0.35", linestyle="--", linewidth=1.1, label="척도 중간값=4")
ax.set_yticks(y_positions, plot_order)
ax.set_xlim(1, 7)
ax.set_xticks(range(1, 8))
ax.set_xlabel("평균 점수 (95% CI)")
ax.set_ylabel("종속변수")
ax.set_title("사용자실험 설문 종속변수의 전체 평균")
ax.grid(axis="x", color="0.88", linewidth=.7)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

fig.savefig(FIGURE_DIR / "figure_02_overall_means_ci.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure_02_overall_means_ci.pdf", bbox_inches="tight")
plt.show()


### 5-2. UI 유형별 종속변수 평균 막대그래프

실제 참가자 점수에서 Baseline과 A2UI의 평균을 계산하고, 각 집단 내 표준오차와 $t$ 임계값으로 95% 신뢰구간을 표시합니다.


In [ ]:
ui_order = ["Baseline", "A2UI"]
ui_colors = {"Baseline": "#337DD1", "A2UI": "#F0642E"}
variable_order = list(DV_ITEMS)
variable_labels = [
    "지각된\n유용성",
    "사용\n용이성",
    "전반적\n만족도",
    "인지\n부하",
    "결과 신뢰도\n·품질",
    "상호작용\n품질",
    "재사용\n의도",
]

ui_bar_statistics = {}
for ui in ui_order:
    ui_scores = scores.loc[groups["UI"].eq(ui), variable_order]
    sample_size = len(ui_scores)
    critical_t_ui = stats.t.ppf(.975, df=sample_size - 1)
    ui_bar_statistics[ui] = {
        "N": sample_size,
        "mean": ui_scores.mean(),
        "ci": critical_t_ui * ui_scores.sem(ddof=1),
    }

x_positions = np.arange(len(variable_order))
bar_width = 0.36
fig, ax = plt.subplots(figsize=(10.2, 5.8))

for offset_index, ui in enumerate(ui_order):
    offset = (offset_index - 0.5) * bar_width
    statistics = ui_bar_statistics[ui]
    ax.bar(
        x_positions + offset,
        statistics["mean"][variable_order],
        width=bar_width,
        yerr=statistics["ci"][variable_order],
        capsize=4,
        color=ui_colors[ui],
        edgecolor="white",
        linewidth=0.6,
        error_kw={"ecolor": "#555555", "elinewidth": 1.2},
        label=f"{ui} (n={statistics['N']})",
        zorder=3,
    )

ax.set_xticks(x_positions, variable_labels)
ax.set_ylim(1, 7)
ax.set_yticks(range(1, 8))
ax.set_ylabel("사후 설문 평균 (1–7점)")
ax.set_xlabel("")
ax.set_title("UI 유형별 사후 설문 종속변수 평균")
ax.grid(axis="y", color="#D9D5CE", linewidth=0.8, alpha=0.8, zorder=0)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=2)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

ui_bar_png = FIGURE_DIR / "figure_02_ui_grouped_bars_ci.png"
ui_bar_pdf = FIGURE_DIR / "figure_02_ui_grouped_bars_ci.pdf"
fig.savefig(ui_bar_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(ui_bar_pdf, bbox_inches="tight", facecolor="white")
plt.show()

ui_bar_means = pd.DataFrame({
    ui: ui_bar_statistics[ui]["mean"] for ui in ui_order
}).reindex(variable_order)
ui_bar_means.index.name = "종속변수"
display(ui_bar_means.round(2))


**그림 3. UI 유형별 사후 설문 7개 종속변수 평균.** 막대는 집단 평균, 오차막대는 95% 신뢰구간이다(각 $n=20$). 인지부하는 점수가 높을수록 부하가 크므로 다른 변수와 해석 방향이 반대이다.


In [ ]:
table_columns = [
    "전체 (N=40)",
    "Baseline (n=20)",
    "A2UI (n=20)",
    "초보자 (n=20)",
    "숙련자 (n=20)",
]
table_data = publication_descriptive_table[table_columns]

fig, ax = plt.subplots(figsize=(12, 6.2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

left, right = 0.025, 0.975
label_width = 0.28
value_width = (right - left - label_width) / 5
x_edges = [left, left + label_width] + [
    left + label_width + value_width * i for i in range(1, 6)
]
x_centers = [(x_edges[i] + x_edges[i + 1]) / 2 for i in range(6)]

ax.text(
    left, 0.965,
    "Table 3. UI 유형 및 숙련도에 따른 사후 설문 기술통계",
    ha="left", va="top", fontsize=14, fontweight="bold", fontfamily=KOREAN_FONT,
)

top_rule_y = 0.89
group_y = 0.845
subheader_y = 0.755
body_top_y = 0.695
row_height = 0.074
bottom_rule_y = body_top_y - row_height * len(table_data)

ax.hlines(top_rule_y, left, right, color="black", linewidth=1.3)
ax.text(x_centers[1], group_y, "전체", ha="center", va="center", fontsize=11, fontfamily=KOREAN_FONT)
ax.text((x_edges[2] + x_edges[4]) / 2, group_y, "UI type", ha="center", va="center", fontsize=11, fontfamily=KOREAN_FONT)
ax.text((x_edges[4] + x_edges[6]) / 2, group_y, "숙련도", ha="center", va="center", fontsize=11, fontfamily=KOREAN_FONT)
ax.hlines(0.812, x_edges[2] + 0.01, x_edges[4] - 0.01, color="0.35", linewidth=0.8)
ax.hlines(0.812, x_edges[4] + 0.01, x_edges[6] - 0.01, color="0.35", linewidth=0.8)

ax.text(x_edges[0] + 0.006, subheader_y, "사후 설문", ha="left", va="center", fontsize=10.5, fontweight="bold", fontfamily=KOREAN_FONT)
subheaders = ["Overall\n$N$ = 40", "Baseline\n$n$ = 20", "A2UI\n$n$ = 20", "초보자\n$n$ = 20", "숙련자\n$n$ = 20"]
for column_index, header in enumerate(subheaders, start=1):
    ax.text(x_centers[column_index], subheader_y, header, ha="center", va="center", fontsize=10, linespacing=1.4, fontfamily=KOREAN_FONT)

ax.hlines(body_top_y, left, right, color="black", linewidth=1.0)
for row_index, (variable, values) in enumerate(table_data.iterrows()):
    y = body_top_y - row_height * (row_index + 0.5)
    ax.text(x_edges[0] + 0.006, y, variable, ha="left", va="center", fontsize=10.2, fontfamily=KOREAN_FONT)
    for column_index, value in enumerate(values, start=1):
        ax.text(x_centers[column_index], y, value, ha="center", va="center", fontsize=10.2, fontfamily=KOREAN_FONT)

ax.hlines(bottom_rule_y, left, right, color="black", linewidth=1.3)
note_y = bottom_rule_y - 0.045
ax.text(
    left, note_y,
    "주. 값은 M (SD)이다. 모든 변수는 7점 척도이며, 인지부하는 점수가 높을수록 부하가 크다.",
    ha="left", va="top", fontsize=9.2, fontfamily=KOREAN_FONT,
)
ax.text(
    left, note_y - 0.042,
    "숙련도는 사전 AI 리터러시 평균 6.0을 기준으로 분류하였다.",
    ha="left", va="top", fontsize=9.2, fontfamily=KOREAN_FONT,
)

table_image_path = FIGURE_DIR / "table_03_post_survey_descriptive_statistics.png"
fig.savefig(table_image_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"표 이미지 저장: {table_image_path}")
